# Feasibility check: does a matching x* always exist? (2D_cond_1D)

This notebook is a diagnostic for a specific reviewer question on the paper:

> I'm curious about the case in which there is not a good x* such that
> P(Y|X=x*) matches the target distribution well. The work in this paper
> appears to assume that the matching is feasible in the first place.

`Exp_2D_cond_1D.ipynb` always builds its target y-distribution as
`P(Y|X=x_star)` for a real `x_star` (see its "GMM PARAMETERS" cell) -- so a
perfect match exists by construction. Here we reuse the exact same joint GMM
and the exact same trained models, but test targets that are **not** of that
form.

All the diagnostic logic lives in `simulations/src/feasibility_check.py`
(same as every other `src/*.py` module in this repo); this notebook just
configures and calls it, following the same structure as `Exp_2D_cond_1D.ipynb`.

Two independent checks per target:
1. **Analytic** (`feasibility_check.diagnose_target`): the exact closed-form
   L2 distance `D(x) = ||P(Y|X=x) - target||^2`, swept over `x`. If
   `min_x D(x)` stays bounded well above the feasible baseline's floor, no
   `x` reproduces the target.
2. **The paper's actual method** (`feasibility_check.run_cross_check`, which
   calls `Optimization.optimize_LGD` unmodified): confirm MLGD / MLGD-F also
   report a non-trivial residual MMD loss on these targets.


## Setup -- reuse the 2D_cond_1D joint GMM and trained models

In [ ]:
import os
# ============================================================
# CONFIG
# EXPERIMENT_NAME must match Exp_2D_cond_1D.ipynb: we reuse its GMM
# parameters (mu_list, Sigma_list, alpha) and its trained checkpoints
# unchanged -- this notebook only adds new *targets* and diagnostics,
# it does not retrain anything.
# ============================================================
EXPERIMENT_NAME   = "2D_cond_1D"
FEAS_NAME         = "2D_feasibility_check"
GLOBAL_SEED       = 42

BASE_DIR = os.path.normpath(os.path.join(os.getcwd(), ".."))
PARAMS_DIR        = f"{BASE_DIR}/params"
CHECKPOINT_DIR    = f"{BASE_DIR}/checkpoints/{EXPERIMENT_NAME}"
RESULTS_DIR       = f"{BASE_DIR}/results/{EXPERIMENT_NAME}"
FEAS_RESULTS_DIR  = f"{BASE_DIR}/results/{FEAS_NAME}"

# Architecture -- must match what Exp_2D_cond_1D.ipynb trained with
NBLOCKS           = 3
NUNITS            = 128
NBLOCKS_CM        = 3
NUNITS_CM         = 128
DIFFUSION_STEPS   = 100
CONDITION_ON      = 1   # dim(x)=1, dim(y)=1

# Optimization (for the MLGD / MLGD-F cross-check section)
N_ATTEMPT_OPTIM_FEAS       = 10   # fewer than Exp_2D_cond_1D's 25: this is a cross-check, not the main result
NSAMPLES_IN_OPTIM_FOR_MMD  = 250
NUM_X_T_LGD                = 3

# Analytic diagnostic
X_GRID_BOUNDS = (-12, 12)   # covers the joint GMM's mean range with margin
Y_GRID        = None        # set below, once we know the y-range to plot


In [ ]:
import os, sys

src_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "src")
src_path = os.path.normpath(src_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"src path on sys.path: {src_path}")


In [ ]:
# Install dependencies if needed
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "flow_matching", "POT", "-q"])


In [ ]:
import os, sys, json
import importlib
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

import Diffusion
import ConsistencyModels
import dist_utils
import experiment_utils
import feasibility_check as fc
from ConsistencyModels import ConsistencyModeliCT

for mod in [Diffusion, ConsistencyModels, dist_utils, experiment_utils, fc]:
    importlib.reload(mod)

os.makedirs(FEAS_RESULTS_DIR, exist_ok=True)
print("Imports done.")


In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Load the 2D_cond_1D GMM parameters

`fc.ensure_gmm_params` loads the parameters saved by `Exp_2D_cond_1D.ipynb`
if available, or regenerates them deterministically (same seed, same code)
if not -- either way, `mu_list`/`Sigma_list`/`alpha` end up identical to what
`model_cond`, `model_uncond`, and the consistency model were trained on, so
this notebook does not require having run `Exp_2D_cond_1D.ipynb` first.


In [ ]:
mu_list, Sigma_list, alpha, x_star = fc.ensure_gmm_params(
    PARAMS_DIR, RESULTS_DIR, EXPERIMENT_NAME, GLOBAL_SEED
)
Y_GRID = np.linspace(-15, 15, 400)

print(f"x_star (baseline target, from Exp_2D_cond_1D) = {x_star}")
print(f"Loaded {len(mu_list)} joint-GMM components.")


## Load trained models (Consistency Model + Diffusion, conditional and unconditional)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(
    nfeatures=1, condition_on=CONDITION_ON, nunits=NUNITS_CM, depth=NBLOCKS_CM
)
if not experiment_utils.load_checkpoint_with_hf_fallback(
    Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR, EXPERIMENT_NAME, GLOBAL_SEED, device
):
    raise RuntimeError("Consistency model checkpoint not found. Train it via Exp_2D_cond_1D.ipynb first.")


In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

model_cond = Diffusion.DiffusionModel(
    nfeatures=2, nblocks=NBLOCKS, nunits=NUNITS,
    condition=True, condition_on=CONDITION_ON, diffusion_steps=DIFFUSION_STEPS
)
if not experiment_utils.load_checkpoint_with_hf_fallback(
    model_cond, "Diffusion_cond", CHECKPOINT_DIR, EXPERIMENT_NAME, GLOBAL_SEED, device
):
    raise RuntimeError("Diffusion_cond checkpoint not found. Train it via Exp_2D_cond_1D.ipynb first.")


In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

model_uncond = Diffusion.DiffusionModel(
    nfeatures=CONDITION_ON, nblocks=NBLOCKS, nunits=NUNITS,
    condition=False, diffusion_steps=DIFFUSION_STEPS
)
if not experiment_utils.load_checkpoint_with_hf_fallback(
    model_uncond, "Diffusion_uncond", CHECKPOINT_DIR, EXPERIMENT_NAME, GLOBAL_SEED, device
):
    raise RuntimeError("Diffusion_uncond checkpoint not found. Train it via Exp_2D_cond_1D.ipynb first.")


## Part A -- Analytic diagnostic (closed form, no trained model needed)

`feasibility_check.diagnose_target` sweeps `x`, computes the exact L2
distance to `target` at every point (via `dist_utils.compute_conditionals` +
`compute_alpha` + `gmm_l2_distance`), plots the curve and a density overlay,
and reports `x*`/`D_min`.

### A0. Calibration: the feasible-by-construction baseline (`P(Y|X=x_star)`, as in Exp_2D_cond_1D)

In [ ]:
target_baseline = fc.target_from_x(mu_list, Sigma_list, alpha, x_star)
diag_baseline = fc.diagnose_target(
    mu_list, Sigma_list, alpha, target_baseline, X_GRID_BOUNDS,
    label="baseline (feasible)", y_grid=Y_GRID, save_dir=FEAS_RESULTS_DIR
)

FEASIBILITY_FLOOR = max(diag_baseline["d_min"], 1e-6) * 50  # "clearly infeasible" threshold, calibrated off this floor
print(f"Feasibility floor for calling a target 'infeasible': D_min > {FEASIBILITY_FLOOR:.3e}")


### A1. Achievable variance range (used to calibrate the variance-shrink target below)

In [ ]:
vmin, vmax, xg_var, avg_vars = fc.achievable_variance_range(mu_list, Sigma_list, alpha, X_GRID_BOUNDS)
print(f"Achievable Var(Y|X=x) over x in {X_GRID_BOUNDS}: [{vmin:.4f}, {vmax:.4f}]")

plt.figure(figsize=(6, 4))
plt.plot(xg_var, avg_vars)
plt.xlabel("x"); plt.ylabel("Var(Y | X=x)")
plt.title("Achievable conditional variance across x")
plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FEAS_RESULTS_DIR, "achievable_variance_range.png"), dpi=150)
plt.show()


### A2. Infeasible target 1 -- 50/50 mixture of two far-apart real conditionals

`target = 0.5 * P(Y|X=-5) + 0.5 * P(Y|X=+5)`. Both halves are individually
real conditionals, but generically no *single* x reproduces their mixture,
because the joint-GMM components' weights shift smoothly (not as a fixed
two-point split) as x moves.

In [ ]:
target_mix = fc.target_mixture_of_two_x(mu_list, Sigma_list, alpha, -5.0, 5.0)
diag_mix = fc.diagnose_target(
    mu_list, Sigma_list, alpha, target_mix, X_GRID_BOUNDS,
    label="case1: mixture of two x", feasibility_floor=FEASIBILITY_FLOOR,
    y_grid=Y_GRID, save_dir=FEAS_RESULTS_DIR
)


### A3. Infeasible target 2 -- variance below the achievable range

Same means/weights as the real conditional at `x=0`, but with every
component variance scaled down until it sits below `vmin` from A1 -- so by
construction no `x` anywhere in the sweep can reach it.

In [ ]:
x_ref = torch.tensor([0.0])
scale = fc.safe_shrink_scale(mu_list, Sigma_list, alpha, x_ref, vmin)
target_var = fc.target_shrink_variance(mu_list, Sigma_list, alpha, x_ref, scale=scale)
print(f"target variance = {scale * fc.mean_variance(*fc.conditional_gmm_at_x(mu_list, Sigma_list, alpha, x_ref))[1]:.4f}  (< vmin = {vmin:.4f})")

diag_var = fc.diagnose_target(
    mu_list, Sigma_list, alpha, target_var, X_GRID_BOUNDS,
    label="case2: variance below achievable range", feasibility_floor=FEASIBILITY_FLOOR,
    y_grid=Y_GRID, save_dir=FEAS_RESULTS_DIR
)


### A4. Infeasible target 3 -- hand-built adversarial bimodal target

An equal mixture of two of the *original joint-GMM components' own*
`(mu_y, Sigma_yy)` (components 0 and 9, on opposite ends of the layout),
independent of any single conditional.

This is still an **infeasibility** case (no real `x` reproduces it) -- "adversarial"
here describes how the target was constructed, not the optimization
landscape. A4a below is the distinct case: a target that *is* reachable,
but whose landscape is hard to search.

In [ ]:
target_custom = fc.target_custom_bimodal(mu_list, Sigma_list, alpha, (0, 9))
diag_custom = fc.diagnose_target(
    mu_list, Sigma_list, alpha, target_custom, X_GRID_BOUNDS,
    label="case3: adversarial bimodal", feasibility_floor=FEASIBILITY_FLOOR,
    y_grid=Y_GRID, save_dir=FEAS_RESULTS_DIR
)


### A4a. Feasible but hard -- a real x* with a decoy local minimum

Distinct from A2-A4: here a perfect match genuinely exists (`D(x*)=0` by
construction, same as the baseline), but `D(x)` also has a secondary local
minimum elsewhere. This tests optimizer robustness rather than feasibility:
does MLGD / MLGD-F's gradient-guided search reliably find the true `x*`, or
does it get pulled into the decoy and converge confidently to the wrong
answer?

`feasibility_check.find_decoy_candidate` sweeps candidate real `x*` values,
builds `target_from_x(x*)` for each (feasible by construction), and looks
for a secondary local minimum in `D(x)` away from `x*` -- we take the most
deceptive one found (smallest decoy `D`).

In [ ]:
decoy_candidates = fc.find_decoy_candidate(mu_list, Sigma_list, alpha, X_GRID_BOUNDS)
print(f"{'x* (real, feasible)':>20} {'decoy x':>10} {'decoy D':>10}")
for x_star_c, decoy_x, decoy_d in decoy_candidates[:8]:
    print(f"{x_star_c:20.3f} {decoy_x:10.3f} {decoy_d:10.4f}")

x_star_hard_val, decoy_x_val, decoy_d_val = decoy_candidates[0]
x_star_hard = torch.tensor([x_star_hard_val])
print(f"\nselected x_star_hard = {x_star_hard_val:.3f}  (decoy at x={decoy_x_val:.3f}, decoy D={decoy_d_val:.4f})")

target_hard = fc.target_from_x(mu_list, Sigma_list, alpha, x_star_hard)
diag_hard = fc.diagnose_target(
    mu_list, Sigma_list, alpha, target_hard, X_GRID_BOUNDS,
    label="A4a: hard landscape (feasible)", y_grid=Y_GRID, save_dir=FEAS_RESULTS_DIR
)
assert diag_hard["d_min"] < 1e-3, "selected x_star_hard is not actually feasible -- pick a different decoy_candidates row"


## Part B -- Cross-check with the paper's actual method (MLGD / MLGD-F)

`feasibility_check.run_cross_check` reruns `Optimization.optimize_LGD` --
unmodified, the same function used in `Exp_2D_cond_1D.ipynb` -- on a target,
once through `model_cond` (MLGD) and once through `Cos_ConsistencyModeliCT`
(MLGD-F). If Part A is right, the reported MMD `final_loss` should plateau
at a non-trivial value on the infeasible targets (never approaching the
near-zero loss seen on the feasible baseline), with the recovered `x`
landing near the analytic `x*` found above.

### B0. Sanity check on the feasible baseline (loss should approach ~0, x should approach x_star)

In [ ]:
df_baseline, summary_baseline = fc.run_cross_check(
    model_uncond, model_cond, Cos_ConsistencyModeliCT, target_baseline, mu_list, Sigma_list, alpha,
    label="baseline (feasible)", global_seed=GLOBAL_SEED, device=device,
    n_attempts=N_ATTEMPT_OPTIM_FEAS, nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, num_x_t=NUM_X_T_LGD
)


### B1-B3. Cross-check on the three infeasible targets

In [ ]:
df_mix, summary_mix = fc.run_cross_check(
    model_uncond, model_cond, Cos_ConsistencyModeliCT, target_mix, mu_list, Sigma_list, alpha,
    label="case1: mixture of two x", global_seed=GLOBAL_SEED, device=device,
    n_attempts=N_ATTEMPT_OPTIM_FEAS, nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, num_x_t=NUM_X_T_LGD
)


In [ ]:
df_var, summary_var = fc.run_cross_check(
    model_uncond, model_cond, Cos_ConsistencyModeliCT, target_var, mu_list, Sigma_list, alpha,
    label="case2: variance below achievable range", global_seed=GLOBAL_SEED, device=device,
    n_attempts=N_ATTEMPT_OPTIM_FEAS, nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, num_x_t=NUM_X_T_LGD
)


In [ ]:
df_custom, summary_custom = fc.run_cross_check(
    model_uncond, model_cond, Cos_ConsistencyModeliCT, target_custom, mu_list, Sigma_list, alpha,
    label="case3: adversarial bimodal", global_seed=GLOBAL_SEED, device=device,
    n_attempts=N_ATTEMPT_OPTIM_FEAS, nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, num_x_t=NUM_X_T_LGD
)


### B4. Cross-check on the feasible-but-hard-landscape target

Unlike B1-B3, a perfect match exists here (`diag_hard["d_min"] ~ 0` at
`diag_hard["x_star"]`). The question is whether `x_recovered` clusters at
that true `x*`, or at the decoy found above -- i.e. whether the optimizer
gets trapped by a good-looking wrong answer even when a better one exists.

In [ ]:
df_hard, summary_hard = fc.run_cross_check(
    model_uncond, model_cond, Cos_ConsistencyModeliCT, target_hard, mu_list, Sigma_list, alpha,
    label="A4a: hard landscape (feasible)", global_seed=GLOBAL_SEED, device=device,
    n_attempts=N_ATTEMPT_OPTIM_FEAS, nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, num_x_t=NUM_X_T_LGD
)

for method, g in df_hard.groupby("method"):
    print(f"\n--- {method}: recovered x per run (true x*={x_star_hard_val:.2f}, decoy at x={decoy_x_val:.2f}) ---")
    print(g.sort_values("final_mmd_loss")[["run", "x_recovered", "final_mmd_loss"]].to_string(index=False))


## Results summary

In [ ]:
comparison_df = pd.DataFrame([
    fc.comparison_row("baseline (feasible)",          diag_baseline, summary_baseline),
    fc.comparison_row("case1: mixture of two x",       diag_mix,      summary_mix),
    fc.comparison_row("case2: variance below range",   diag_var,      summary_var),
    fc.comparison_row("case3: adversarial bimodal",    diag_custom,   summary_custom),
    fc.comparison_row("A4a: hard landscape (feasible)", diag_hard, summary_hard),
]).set_index("case")
display(comparison_df)

path = os.path.join(FEAS_RESULTS_DIR, f"{FEAS_NAME}_results_seed{GLOBAL_SEED}.json")
with open(path, "w") as f:
    json.dump({
        "experiment": FEAS_NAME,
        "seed": GLOBAL_SEED,
        "feasibility_floor": FEASIBILITY_FLOOR,
        "comparison": comparison_df.reset_index().to_dict(orient="records"),
        "curves": {
            d["label"]: {"x": d["x_grid"], "D": d["d_grid"]}
            for d in [diag_baseline, diag_mix, diag_var, diag_custom, diag_hard]
        },
    }, f, indent=2)
print(f"Results saved to {path}")


## Takeaway

- **Baseline** (target constructed as `P(Y|X=x_star)`): `D(x)` reaches ~0 at
  `x = x_star`, and MLGD/MLGD-F both converge to a near-zero MMD loss at
  `x \approx x_star` -- this is the paper's existing setting, and it is
  feasible by construction.
- **Cases 1-3** (targets not of that form): `D(x)` never approaches 0 -- it
  bottoms out at a level far above the baseline floor -- and MLGD/MLGD-F
  independently confirm this: their residual MMD loss plateaus at a
  comparably non-trivial value instead of vanishing, converging to the same
  region of x as the analytic `x*`.

This gives a direct, reusable diagnostic for the reviewer's question:
`feasibility_check.py` quantifies how far the *best possible* x is from the
target, independent of whether the target was designed to be reachable. The
gap is not a training artifact -- it shows up identically in the closed-form
GMM math and in the learned diffusion / consistency models.